In [1]:
# ────────────────────────────────────────────────────────────────
# Script: anonimizar_personas.py
# Autor: Jimmy Suárez
# Descripción:
#   Genera un catálogo único de personas basado en columnas sensibles
#   (Autor, Especialista, Cliente, Jefe), incluyendo los correos electrónicos
#   asociados al campo Email_Cliente. Reemplaza estos valores por códigos
#   anónimos y correos ficticios en el dataset original. Guarda los resultados
#   para uso privado y análisis posterior.
# ────────────────────────────────────────────────────────────────

In [2]:
import pandas as pd

In [3]:
# ────────────────────────────────────────────────────────────────
# 1. Cargar archivo original
# ────────────────────────────────────────────────────────────────

ruta_original = r"C:\Users\jimmy\Documents\UNIR\TFM\BD_ARANDA.xlsx"
ruta_salida_anon = r"C:\Users\jimmy\Documents\UNIR\TFM\BD_ARANDA_anon.xlsx"
ruta_catalogo = r"C:\Users\jimmy\Documents\UNIR\TFM\catalogo_personas.xlsx"

df = pd.read_excel(ruta_original)
print("✅ Dataset original cargado correctamente.")

✅ Dataset original cargado correctamente.


In [4]:
# ────────────────────────────────────────────────────────────────
# 2. Crear catálogo unificado de personas
# ────────────────────────────────────────────────────────────────

columnas_personales = ['Autor', 'Especialista', 'Cliente', 'Jefe']

# Unir valores únicos de todas las columnas relevantes
personas_unicas = pd.Series(dtype="str")
for col in columnas_personales:
    if col in df.columns:
        personas_unicas = pd.concat([personas_unicas, df[col].dropna().astype(str)])

personas_unicas = personas_unicas.drop_duplicates().reset_index(drop=True)

# Asignar códigos anónimos únicos
catalogo = pd.DataFrame({
    'Valor_Original': personas_unicas,
    'Codigo_Anonimo': ['Persona_' + str(i).zfill(4) for i in range(1, len(personas_unicas) + 1)]
})

print(f"🔐 Catálogo generado con {len(catalogo)} personas únicas.")


🔐 Catálogo generado con 708 personas únicas.


In [5]:
# ────────────────────────────────────────────────────────────────
# 3. Incluir email_original y generar email_anonimo
# ────────────────────────────────────────────────────────────────

if 'Cliente' in df.columns and 'Email_Cliente' in df.columns:
    emails = df[['Cliente', 'Email_Cliente']].dropna().drop_duplicates()
    emails.columns = ['Valor_Original', 'email_original']
    catalogo = catalogo.merge(emails, on='Valor_Original', how='left')
    catalogo['email_anonimo'] = catalogo['Codigo_Anonimo'].str.lower() + '@emailanon.com'
    print("📧 Emails anonimizados y agregados al catálogo.")

📧 Emails anonimizados y agregados al catálogo.


In [6]:
# ────────────────────────────────────────────────────────────────
# 4. Reemplazar los valores originales por los códigos en el dataset
# ────────────────────────────────────────────────────────────────

# Reemplazo de columnas personales
mapa_anon = dict(zip(catalogo['Valor_Original'], catalogo['Codigo_Anonimo']))
for col in columnas_personales:
    if col in df.columns:
        df[col] = df[col].astype(str).map(mapa_anon)
        print(f"✔️ Columna '{col}' anonimizada.")

# Reemplazo del email del cliente
if 'Email_Cliente' in df.columns:
    mapa_email = dict(zip(catalogo['email_original'], catalogo['email_anonimo']))
    df['Email_Cliente'] = df['Email_Cliente'].map(mapa_email)
    print("✔️ Columna 'Email_Cliente' anonimizada.")

✔️ Columna 'Autor' anonimizada.
✔️ Columna 'Especialista' anonimizada.
✔️ Columna 'Cliente' anonimizada.
✔️ Columna 'Jefe' anonimizada.
✔️ Columna 'Email_Cliente' anonimizada.


In [7]:
# ────────────────────────────────────────────────────────────────
# 5. Guardar archivos finales
# ────────────────────────────────────────────────────────────────

df.to_excel(ruta_salida_anon, index=False)
catalogo.to_excel(ruta_catalogo, index=False)

print("\n💾 Archivo anonimizado guardado en:")
print("   ", ruta_salida_anon)
print("📁 Catálogo de personas guardado en:")
print("   ", ruta_catalogo)


💾 Archivo anonimizado guardado en:
    C:\Users\jimmy\Documents\UNIR\TFM\BD_ARANDA_anon.xlsx
📁 Catálogo de personas guardado en:
    C:\Users\jimmy\Documents\UNIR\TFM\catalogo_personas.xlsx


In [8]:
# ────────────────────────────────────────────────────────────────
# 6. Validación de la anonimización
# ────────────────────────────────────────────────────────────────

print("\n🔎 Iniciando validación de anonimización...")

# 6.1. Verificar que no quedan valores originales en las columnas sensibles
errores_restantes = {}
for col in columnas_personales:
    valores_unicos = df[col].dropna().unique()
    if not all(str(v).startswith('Persona_') for v in valores_unicos):
        errores_restantes[col] = [v for v in valores_unicos if not str(v).startswith('Persona_')]

if errores_restantes:
    print("❌ ERROR: Se encontraron valores no anonimizados en las siguientes columnas:")
    for col, valores in errores_restantes.items():
        print(f"   - {col}: {valores[:5]}...")  # Muestra hasta 5 ejemplos
else:
    print("✅ Todas las columnas sensibles han sido correctamente anonimizadas.")

# 6.1.b Verificar formato de correos anonimizados
if 'Email_Cliente' in df.columns:
    correos_unicos = df['Email_Cliente'].dropna().unique()
    if not all(str(c).endswith('@emailanon.com') for c in correos_unicos):
        print("❌ ERROR: Se encontraron correos no anonimizados en 'Email_Cliente'.")
        print([c for c in correos_unicos if not str(c).endswith('@emailanon.com')][:5])
    else:
        print("✅ Todos los correos en 'Email_Cliente' han sido correctamente anonimizados.")

# 6.2. Verificar que todos los códigos existen en el catálogo
codigos_catalogo = set(catalogo['Codigo_Anonimo'])
codigos_dataset = set()

for col in columnas_personales:
    codigos_dataset.update(df[col].dropna().unique())

diferencia = codigos_dataset - codigos_catalogo
if diferencia:
    print("❌ ERROR: Se encontraron códigos en el dataset que no existen en el catálogo:")
    print(diferencia)
else:
    print("✅ Todos los códigos anonimizados en el dataset existen en el catálogo.")

# 6.3. Validar consistencia de asignación (frecuencia de uso de códigos)
ejemplo_reversion = df[columnas_personales].stack().value_counts().head(3)
print("\n🧪 Ejemplo de códigos anonimizados más frecuentes:")
print(ejemplo_reversion)



🔎 Iniciando validación de anonimización...
✅ Todas las columnas sensibles han sido correctamente anonimizadas.
✅ Todos los correos en 'Email_Cliente' han sido correctamente anonimizados.
✅ Todos los códigos anonimizados en el dataset existen en el catálogo.

🧪 Ejemplo de códigos anonimizados más frecuentes:
Persona_0005    2771
Persona_0013    2766
Persona_0002    2300
Name: count, dtype: int64
